# 02. Preprocesamiento de Imágenes y Extracción de Características
## Clasificación del Grado de Artrosis de Rodilla mediante Machine Learning

**Asignatura:** Machine Learning 1 (23433)  
**Carrera:** Ingeniería Electrónica - FIUNA  

Este notebook documenta y ejecuta el pipeline de preprocesamiento de imágenes médicas e ingeniería de características para la clasificación del grado de artrosis de rodilla (escala Kellgren-Lawrence de 0 a 4).

### Puntos clave de la metodología
1. **Proporción conservada:** cada imagen se ajusta a un canvas de 256×256 con relleno negro, sin estirar la anatomía.
2. **Separación bilateral:** las vistas panorámicas se dividen en rodilla izquierda y derecha, y se procesan por separado.
3. **Realce CLAHE:** mejora el contraste local dentro del área radiográfica; el relleno queda negro.
4. **Descriptores:** HOG, gradientes de Sobel, Laplaciano, histograma y estadísticas espaciales.

El relleno negro ocupa parte del canvas y es intencional: evita deformar las radiografías. La resolución fija facilita la comparación computacional, pero no recupera información que no estaba en la imagen original.


In [1]:
import os
import json
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Rutas del proyecto
PROJECT_DIR = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data" / "processed").exists()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError("No se encontró la raíz del proyecto.")

RAW_DIR = PROJECT_DIR / "data" / "raw" / "Training"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURES_DIR = PROJECT_DIR / "reports" / "figures"

print(f"Directorio del proyecto: {PROJECT_DIR}")
print(f"Directorio de procesados: {PROCESSED_DIR}")


Directorio del proyecto: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla
Directorio de procesados: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\data\processed


## 1. Carga y Verificación de las Particiones del Dataset

Cargamos el archivo `dataset_splits.csv` generado durante la fase de análisis exploratorio (EDA) para asegurar la integridad de las particiones de entrenamiento, validación y prueba.

In [2]:
splits_path = PROCESSED_DIR / "dataset_splits.csv"
df_splits = pd.read_csv(splits_path)

def resolve_path(p: str) -> str:
    path_obj = Path(p)
    if path_obj.exists():
        return str(path_obj)
    parts = path_obj.parts
    if "Training" in parts:
        idx = parts.index("Training")
        return str(PROJECT_DIR / "data" / "raw" / Path(*parts[idx:]))
    return str(PROJECT_DIR / "data" / "raw" / "Training" / path_obj.parent.name / path_obj.name)

df_splits["path"] = df_splits["path"].apply(resolve_path)

print(f"Total de registros: {len(df_splits)}")
print("Distribución por partición:")
print(df_splits["split_final"].value_counts())
print("\nDistribución de resoluciones:")
print(df_splits["resolution"].value_counts())

Total de registros: 1650
Distribución por partición:
split_final
train         1155
test           248
validation     247
Name: count, dtype: int64

Distribución de resoluciones:
resolution
300 x 162    1464
640 x 161     186
Name: count, dtype: int64


## 2. Acondicionamiento anatómico: proporción conservada y vistas bilaterales

Cada vista se escala uniformemente para caber en un canvas de 256×256. El relleno negro conserva la forma original; no se estira la radiografía ni se recorta la rodilla. En imágenes panorámicas de ancho igual o superior a 500 píxeles, se separan las mitades y se representa cada rodilla en su propio canvas.

CLAHE se aplica a la región radiográfica. Las vistas bilaterales se muestran lado a lado en una imagen de 512×256 para conservar la forma de ambos canvases.


In [3]:
# Importación robusta: funciona aunque Jupyter abra el kernel desde otra carpeta.
import sys
from pathlib import Path

project_candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (candidate for candidate in project_candidates if (candidate / "src" / "preprocessing.py").exists()),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "No se encontró src/preprocessing.py. Abrí en Antigravity la carpeta raíz "
        "'proyecto-rodilla' y volvé a ejecutar esta celda."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
import src.preprocessing as preprocessing
preprocessing = importlib.reload(preprocessing)

from src.preprocessing import (
    preprocess_knee_image,
    preprocess_view,
    resize_with_padding,
    split_bilateral,
)



Comparamos las radiografías originales con vistas acondicionadas. El relleno negro es intencional: conserva la proporción, sin estirar ni recortar la anatomía. Las panorámicas se muestran como dos vistas lado a lado.

In [4]:
samples_demo = [
    df_splits[(df_splits["target"] == 0) & (df_splits["resolution"] == "300 x 162")].iloc[0],
    df_splits[(df_splits["target"] == 1) & (df_splits["resolution"] == "300 x 162")].iloc[0],
    df_splits[(df_splits["target"] == 2) & (df_splits["resolution"] == "640 x 161")].iloc[0],
    df_splits[(df_splits["target"] == 3) & (df_splits["resolution"] == "640 x 161")].iloc[0],
    df_splits[(df_splits["target"] == 4) & (df_splits["resolution"] == "300 x 162")].iloc[0],
]

fig, axes = plt.subplots(5, 2, figsize=(11, 14), dpi=150)
for idx, sample in enumerate(samples_demo):
    img = cv2.imread(sample["path"], cv2.IMREAD_GRAYSCALE)
    conditioned_img = preprocess_knee_image(img, target_size=None)
    
    # Original
    axes[idx, 0].imshow(img, cmap="gray")
    axes[idx, 0].set_title(f"Original: {sample['class_name']} ({sample['resolution']})", fontsize=10)
    axes[idx, 0].axis("off")
    
    # Acondicionada
    axes[idx, 1].imshow(conditioned_img, cmap="gray")
    desc = "Dos vistas separadas; proporción conservada + CLAHE" if img.shape[1] >= 500 else "Proporción original + CLAHE"
    axes[idx, 1].set_title(desc, fontsize=10)
    axes[idx, 1].axis("off")

plt.suptitle("Acondicionamiento visual: Proporción original + CLAHE", fontsize=12, fontweight="bold", y=0.99)
plt.tight_layout()

demo_fig_path = FIGURES_DIR / "acondicionamiento_radiografias.png"
plt.savefig(demo_fig_path, dpi=300, bbox_inches="tight")
print(f"Figura guardada en: {demo_fig_path}")
plt.show()






Figura guardada en: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\reports\figures\acondicionamiento_radiografias.png
02_Preprocesamiento.ipynb:cell8:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 4. Extracción de Descriptores de Forma: Dalal-Triggs HOG

Implementamos el descriptor HOG nativo en OpenCV:
- Imagen redimensionada a $128 \times 128$ píxeles.
- Celdas de $32 \times 32$ píxeles y bloques de $2 \times 2$ celdas con normalización $L_2$.
- 8 bins angulares ($0^\circ$ a $180^\circ$).
- Retorna $3 \times 3 \times 4 \times 8 = 288$ características.

In [5]:
from src.features import compute_hog_features


## 5. Pipeline de características visuales (320 descriptores)

El pipeline de extracción genera un vector representativo estructurado en:
1. **Estadísticos globales (5):** media, desviación estándar, mediana, percentiles 25 y 75.
2. **Histograma normalizado (16):** distribución de intensidades radiográficas.
3. **Pirámide espacial 2×2 (8):** media y desviación estándar por cuadrante.
4. **Gradientes de Sobel y Laplaciano (3):** magnitud y variación de bordes.
5. **HOG (288):** histogramas de gradientes orientados.






In [6]:
from src.features import extract_single_knee_features, extract_image_features


## 6. Extracción Masiva del Dataset y Guardado Tabular

Procesamos las $1.650$ radiografías del dataset para consolidar la matriz de características $\mathbf{X}$ y el vector de etiquetas $\mathbf{y}$, exportando los resultados en formato tabular (`features_dataset.csv`) y en matrices binarias comprimidas (`features_arrays.npz`).

In [7]:
rows = []
all_flats = []

print("Extrayendo características para las 1650 radiografías...")
t0 = time.time()

for idx, row in df_splits.iterrows():
    img_gray = cv2.imread(row["path"], cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        raise ValueError(f"No se pudo cargar la imagen: {row['path']}")
    
    feats, _, flat = extract_image_features(img_gray, target_size=(256, 256))
    
    # Metadatos del registro
    feats["path"] = row["path"]
    feats["class_name"] = row["class_name"]
    feats["class_label"] = row["class_label"]
    feats["target"] = int(row["target"])
    feats["split_final"] = row["split_final"]
    feats["resolution_original"] = row["resolution"]
    
    rows.append(feats)
    all_flats.append(flat)

dt = time.time() - t0
print(f"Extracción completada en {dt:.2f} s ({dt/len(df_splits)*1000:.2f} ms/imagen)")

df_features = pd.DataFrame(rows)
print(f"Dimensiones de df_features: {df_features.shape}")

# Guardado en CSV
features_csv_path = PROCESSED_DIR / "features_dataset.csv"
df_features.to_csv(features_csv_path, index=False)
print(f"Matriz guardada en CSV: {features_csv_path} ({features_csv_path.stat().st_size / (1024*1024):.2f} MB)")

# Guardado en NPZ
features_npz_path = PROCESSED_DIR / "features_arrays.npz"
feature_cols = [
    c for c in df_features.columns
    if c not in ["path", "class_name", "class_label", "target", "split_final", "resolution_original"]
]
X_tabular = df_features[feature_cols].values.astype(np.float32)
X_flat_array = np.array(all_flats, dtype=np.float32)
y_all = df_features["target"].values.astype(np.int64)

train_mask = (df_features["split_final"] == "train").values
val_mask = (df_features["split_final"] == "validation").values
test_mask = (df_features["split_final"] == "test").values

np.savez_compressed(
    features_npz_path,
    X_train_tabular=X_tabular[train_mask],
    X_val_tabular=X_tabular[val_mask],
    X_test_tabular=X_tabular[test_mask],
    X_train_flat=X_flat_array[train_mask],
    X_val_flat=X_flat_array[val_mask],
    X_test_flat=X_flat_array[test_mask],
    y_train=y_all[train_mask],
    y_val=y_all[val_mask],
    y_test=y_all[test_mask],
    feature_names=np.array(feature_cols),
)
print(f"Matrices guardadas en NPZ: {features_npz_path} ({features_npz_path.stat().st_size / (1024*1024):.2f} MB)")
feature_names_path = PROCESSED_DIR / "feature_names.json"
with open(feature_names_path, "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2, ensure_ascii=False)
print(f"Nombres de características guardados: {feature_names_path}")

# Actualización de archivos individuales para compatibilidad
np.savez_compressed(PROCESSED_DIR / "features_train.npz", X=X_tabular[train_mask], y=y_all[train_mask])
np.savez_compressed(PROCESSED_DIR / "features_validation.npz", X=X_tabular[val_mask], y=y_all[val_mask])
np.savez_compressed(PROCESSED_DIR / "features_test.npz", X=X_tabular[test_mask], y=y_all[test_mask])
print("Archivos individuales train/validation/test sincronizados.")


Extrayendo características para las 1650 radiografías...
Extracción completada en 12.42 s (7.53 ms/imagen)
Dimensiones de df_features: (1650, 326)
Matriz guardada en CSV: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\data\processed\features_dataset.csv (9.83 MB)
Matrices guardadas en NPZ: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\data\processed\features_arrays.npz (6.36 MB)
Nombres de características guardados: C:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\ML1\proyecto-rodilla\data\processed\feature_names.json
Archivos individuales train/validation/test sincronizados.


In [8]:
# Inspección de las primeras columnas del DataFrame de características
display(df_features[["path", "class_name", "target", "intensity_mean", "grad_mean", "laplacian_var"]].head())


                                                path  ... laplacian_var
0  c:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\...  ...    379.605255
1  c:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\...  ...    423.432576
2  c:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\...  ...    559.461483
3  c:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\...  ...    501.943021
4  c:\Users\fliaP\Documents\Fiuna\7_semestre\ML1\...  ...    336.798796

[5 rows x 6 columns]


## 7. Conclusiones y próximos pasos

- **Geometría conservada:** se usa ajuste proporcional con relleno negro para evitar estirar o recortar las radiografías.
- **Vistas bilaterales separadas:** cada lado se acondiciona en su propio canvas; para modelos clásicos las características de ambos lados se promedian.
- **Sin variables de resolución:** se excluyen la relación de aspecto y el factor de relleno para reducir el riesgo de que el modelo aprenda el formato del archivo en lugar de señales radiográficas.
- **Próximo paso:** volver a ejecutar este notebook y luego `03_Baseline.ipynb`, `04_Modelos_Avanzados.ipynb` y `05_Resultados.ipynb`. Los resultados anteriores corresponden al preprocesamiento previo y deben recalcularse.

